# Two-branch, GPU TCN — Las Vegas → Phoenix cross-city transfer

This notebook uses the **raw CSVs** directly. A city-specific calendar/trend neural baseline predicts normal log-demand; a GPU TCN trained only on Las Vegas predicts the weather-driven residual. The Phoenix baseline is calibrated only on its pre-test history, while the learned weather relationship is transferred from Las Vegas to Phoenix.

Timeline: **2016–2021 training**, **2022 validation**, **2023 final test**. Newer training examples receive greater weight through a two-year exponential half-life.

In [1]:
# Run first. Paths are Windows-friendly and results are saved only under RESULT_DIR.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
from pathlib import Path
import json, platform, sys
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

DATA_DIR = (Path.cwd().parent / 'data')
MODEL_DIR = Path.cwd()
RESULT_DIR = MODEL_DIR / 'two_branch_tcn_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CITY = 'las_vegas'
TARGET_CITY = 'phoenix'
TRAIN_END = pd.Timestamp('2021-12-31 23:00:00')
VALID_END = pd.Timestamp('2022-12-31 23:00:00')
LOOKBACK = 168
HORIZON = 1
HALF_LIFE_YEARS = 2.0
BASELINE_EPOCHS, TCN_EPOCHS, BATCH_SIZE = 80, 100, 64
SEED = 42
USE_MIXED_PRECISION = True
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
assert DATA_DIR.exists(), f'Raw-data folder not found: {DATA_DIR}'
assert SOURCE_CITY != TARGET_CITY
print('Results will be saved in:', RESULT_DIR)

Results will be saved in: D:\Downloads\BTP\model_training\lstm_training_3\neural_models\two_branch_tcn_results


## Required GPU and package sanity check

Training stops if TensorFlow does not execute a real matrix operation on GPU:0. The data reader may use CPU, but the neural-network forward pass, gradients, and optimizer updates use the GPU.

In [2]:
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)
print('joblib:', joblib.__version__, '| TensorFlow:', tf.__version__, '| Keras:', tf.keras.__version__)
print('Built with CUDA:', tf.test.is_built_with_cuda())
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('NO GPU DETECTED — training is intentionally stopped. Use a CUDA-enabled TensorFlow environment and restart the kernel.')
for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc: print('Memory-growth note:', exc)
tf.config.set_soft_device_placement(False)
print('Physical GPUs:', gpus)
for gpu in gpus: print('GPU details:', tf.config.experimental.get_device_details(gpu))
with tf.device('/GPU:0'):
    probe = tf.linalg.matmul(tf.ones((512, 512)), tf.ones((512, 512)))
assert 'GPU' in probe.device.upper(), f'GPU probe ran on {probe.device}'
print('GPU execution verified:', probe.device)
del probe
if USE_MIXED_PRECISION: tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Precision policy:', tf.keras.mixed_precision.global_policy())
print('SANITY CHECK PASSED — model operations are GPU-enforced.')

Python: 3.10.20
Platform: Windows-10-10.0.26200-SP0
NumPy: 1.26.4 | pandas: 2.3.3 | scikit-learn: 1.7.2
joblib: 1.5.3 | TensorFlow: 2.10.0 | Keras: 2.10.0
Built with CUDA: True
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU details: {'device_name': 'NVIDIA GeForce RTX 3050 6GB Laptop GPU', 'compute_capability': (8, 6)}
GPU execution verified: /job:localhost/replica:0/task:0/device:GPU:0
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU, compute capability 8.6
Precision policy: <Policy "mixed_float16">
SANITY CHECK PASSED — model operations are GPU-enforced.


In [3]:
# Load, timestamp-align, clean, and engineer weather/calendar inputs. No future fill is used.
RANGES = {'temperature_c':(-60,60),'dew_point_c':(-80,50),'relative_humidity_percent':(0,100),'station_pressure_kpa':(80,110),'wind_direction_deg':(0,360),'wind_speed_ms':(0,75),'horizontal_visibility_km':(0,100),'precipitable_water_mm':(0,100),'hourly_precipitation_cm':(0,100),'total_sky_cover_tenths':(0,10),'opaque_sky_cover_tenths':(0,10),'extraterrestrial_horizontal_radiation_wh_m2':(0,np.inf),'extraterrestrial_direct_normal_radiation_wh_m2':(0,np.inf)}
def load_city(city):
    e = pd.read_csv(DATA_DIR / f'{city}_hourly_energy.csv', parse_dates=['timestamp'])[['timestamp','hourly_load_mw']]
    w = pd.read_csv(DATA_DIR / f'{city}_hourly_weather_2016_2023.csv', parse_dates=['timestamp'])
    assert e.timestamp.notna().all() and w.timestamp.notna().all()
    e = e.sort_values('timestamp').drop_duplicates('timestamp').set_index('timestamp').asfreq('h')
    cols = [c for c in RANGES if c in w]
    w = w.sort_values('timestamp').drop_duplicates('timestamp').set_index('timestamp')[cols].asfreq('h')
    for c in cols:
        w[c] = pd.to_numeric(w[c], errors='coerce')
        lo, hi = RANGES[c]; w.loc[~w[c].between(lo, hi), c] = np.nan
    w = w.ffill()  # past-only; original data has 48 absent weather hours
    df = e.join(w, how='left').reset_index().sort_values('timestamp').reset_index(drop=True)
    df['load'] = pd.to_numeric(df.hourly_load_mw, errors='coerce').where(lambda s: s > 0)
    t = df.timestamp
    df['hour_sin'], df['hour_cos'] = np.sin(2*np.pi*t.dt.hour/24), np.cos(2*np.pi*t.dt.hour/24)
    df['dow_sin'], df['dow_cos'] = np.sin(2*np.pi*t.dt.dayofweek/7), np.cos(2*np.pi*t.dt.dayofweek/7)
    df['month_sin'], df['month_cos'] = np.sin(2*np.pi*(t.dt.month-1)/12), np.cos(2*np.pi*(t.dt.month-1)/12)
    df['is_weekend'] = (t.dt.dayofweek >= 5).astype('float32')
    df['years_since_start'] = (t - t.min()).dt.total_seconds() / (365.25*24*3600)
    rad = np.deg2rad(df['wind_direction_deg']); df['wind_dir_sin'], df['wind_dir_cos'] = np.sin(rad), np.cos(rad)
    df['cooling_degree_c'] = np.clip(df.temperature_c-18, 0, None); df['heating_degree_c'] = np.clip(18-df.temperature_c, 0, None)
    df['temp_x_humidity'] = df.temperature_c * df.relative_humidity_percent
    weather = [c for c in cols if c != 'wind_direction_deg'] + ['wind_dir_sin','wind_dir_cos','cooling_degree_c','heating_degree_c','temp_x_humidity']
    assert df.timestamp.diff().dropna().eq(pd.Timedelta(hours=1)).all()
    return df, weather

source_df, WEATHER_FEATURES = load_city(SOURCE_CITY)
target_df, target_weather = load_city(TARGET_CITY)
assert WEATHER_FEATURES == target_weather, 'Cities do not have matching weather features.'
print(SOURCE_CITY, source_df.shape, '|', TARGET_CITY, target_df.shape, '| weather features:', len(WEATHER_FEATURES))

las_vegas (70128, 29) | phoenix (70128, 29) | weather features: 17


In [4]:
# Baseline branch: calendar + long-run trend only. It deliberately receives no weather variables.
BASELINE_FEATURES = ['hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos','is_weekend','years_since_start']
def masks(df):
    return (df.timestamp <= TRAIN_END), ((df.timestamp > TRAIN_END) & (df.timestamp <= VALID_END)), (df.timestamp > VALID_END)
def recency_weights(timestamps):
    age_years = (TRAIN_END - timestamps).dt.total_seconds().clip(lower=0) / (365.25*24*3600)
    return np.exp(-np.log(2)*age_years/HALF_LIFE_YEARS).astype('float32')
def build_baseline(n_features):
    x = tf.keras.Input((n_features,), name='calendar_and_trend')
    z = tf.keras.layers.Dense(64, activation='relu')(x)
    z = tf.keras.layers.Dropout(.10)(z)
    z = tf.keras.layers.Dense(32, activation='relu')(z)
    return tf.keras.Model(x, tf.keras.layers.Dense(1, dtype='float32', name='scaled_log_baseline')(z), name='city_baseline_mlp')
def fit_baseline(df, city):
    tr, va, _ = masks(df); usable = df[BASELINE_FEATURES+['load']].notna().all(axis=1)
    tr, va = tr & usable, va & usable
    xsc, ysc = StandardScaler(), StandardScaler()
    Xtr = xsc.fit_transform(df.loc[tr, BASELINE_FEATURES]); Xva = xsc.transform(df.loc[va, BASELINE_FEATURES])
    ytr = ysc.fit_transform(np.log(df.loc[tr,['load']])).ravel(); yva = ysc.transform(np.log(df.loc[va,['load']])).ravel()
    with tf.device('/GPU:0'):
        model = build_baseline(len(BASELINE_FEATURES)); model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    cbs=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)]
    # No device wrapper around fit: tf.data/Keras bookkeeping has legitimate CPU-only kernels.
    model.fit(Xtr,ytr,sample_weight=recency_weights(df.loc[tr,'timestamp']),validation_data=(Xva,yva),epochs=BASELINE_EPOCHS,batch_size=256,verbose=0,callbacks=cbs)
    valid = df[BASELINE_FEATURES].notna().all(axis=1); pred=np.full(len(df),np.nan)
    pred[valid] = ysc.inverse_transform(model.predict(xsc.transform(df.loc[valid,BASELINE_FEATURES]),verbose=0)).ravel()
    out=df.copy(); out['baseline_log']=pred; out['residual_log']=np.log(out['load'])-out['baseline_log']
    city_dir=RESULT_DIR/city; city_dir.mkdir(exist_ok=True); model.save(city_dir/'baseline_mlp.keras'); joblib.dump(xsc,city_dir/'baseline_feature_scaler.joblib'); joblib.dump(ysc,city_dir/'baseline_target_scaler.joblib')
    return out, model, xsc, ysc

source_df, source_baseline, _, _ = fit_baseline(source_df, SOURCE_CITY)
target_df, target_baseline, _, _ = fit_baseline(target_df, TARGET_CITY)
print("Baselines trained independently using only each city historical 2016-2021 data; 2022 was validation.")

Baselines trained independently using only each city historical 2016-2021 data; 2022 was validation.


In [5]:
# Build causal residual-history features. At hour t they use no value after t.
RESIDUAL_HISTORY = ['resid_lag_0h','resid_lag_1h','resid_lag_24h','resid_lag_168h','resid_roll_mean_24h','resid_roll_std_24h']
def add_residual_history(df):
    out=df.copy(); r=out.residual_log
    out['resid_lag_0h']=r
    out['resid_lag_1h']=r.shift(1); out['resid_lag_24h']=r.shift(24); out['resid_lag_168h']=r.shift(168)
    out['resid_roll_mean_24h']=r.shift(1).rolling(24,min_periods=24).mean()
    out['resid_roll_std_24h']=r.shift(1).rolling(24,min_periods=24).std()
    return out
source_df, target_df = add_residual_history(source_df), add_residual_history(target_df)
TCN_FEATURES = WEATHER_FEATURES + RESIDUAL_HISTORY
tr, va, _ = masks(source_df)
fit_rows = tr & source_df[TCN_FEATURES+['residual_log']].notna().all(axis=1)
x_scaler = StandardScaler().fit(source_df.loc[fit_rows,TCN_FEATURES])
r_scaler = StandardScaler().fit(source_df.loc[fit_rows,['residual_log']])
def scaled_matrix(df):
    X=x_scaler.transform(df[TCN_FEATURES]); y=r_scaler.transform(df[['residual_log']]).ravel(); return X.astype('float32'), y.astype('float32')
X_source, y_source = scaled_matrix(source_df); X_target, y_target = scaled_matrix(target_df)
print('TCN learns residuals, not MW. Feature/target scalers are fitted on Las Vegas training data only.')

TCN learns residuals, not MW. Feature/target scalers are fitted on Las Vegas training data only.


In [6]:
# Memory-efficient sliding-window datasets; a target at p uses inputs [p-LOOKBACK, ..., p-1].
def valid_positions(df, X, y, split):
    tr, va, te = masks(df); mask={'train':tr,'validation':va,'test':te}[split].to_numpy()
    finite=np.isfinite(X).all(axis=1) & np.isfinite(y); bad=np.concatenate(([0],np.cumsum(~finite)))
    positions=[]
    for p in np.flatnonzero(mask):
        if p >= LOOKBACK and bad[p+1]-bad[p-LOOKBACK] == 0: positions.append(p)
    return np.asarray(positions,dtype=int)
def make_dataset(X,y,positions,training=False,sample_weights=None):
    def gen():
        ids=positions.copy()
        if training: np.random.default_rng(SEED).shuffle(ids)
        for start in range(0,len(ids),BATCH_SIZE):
            p=ids[start:start+BATCH_SIZE]
            xb=np.stack([X[i-LOOKBACK:i] for i in p]).astype('float32')
            yb=y[p].astype('float32')
            if sample_weights is None: yield xb, yb
            else: yield xb, yb, sample_weights[p].astype('float32')
    xs=tf.TensorSpec((None,LOOKBACK,len(TCN_FEATURES)),tf.float32); ys=tf.TensorSpec((None,),tf.float32)
    sig=(xs,ys) if sample_weights is None else (xs,ys,tf.TensorSpec((None,),tf.float32))
    return tf.data.Dataset.from_generator(gen,output_signature=sig).prefetch(tf.data.AUTOTUNE)
source_train_pos=valid_positions(source_df,X_source,y_source,'train'); source_val_pos=valid_positions(source_df,X_source,y_source,'validation'); target_test_pos=valid_positions(target_df,X_target,y_target,'test')
assert len(source_train_pos) and len(source_val_pos) and len(target_test_pos)
source_train_weights=recency_weights(source_df.timestamp).to_numpy(dtype='float32')
train_ds=make_dataset(X_source,y_source,source_train_pos,True,sample_weights=source_train_weights); val_ds=make_dataset(X_source,y_source,source_val_pos); target_test_ds=make_dataset(X_target,y_target,target_test_pos)
print('Sequences — LV train:',len(source_train_pos),'LV validation:',len(source_val_pos),'Phoenix 2023 test:',len(target_test_pos))

Sequences — LV train: 52272 LV validation: 8760 Phoenix 2023 test: 8760


In [7]:
def residual_block(x,filters,dilation):
    shortcut=x
    for _ in range(2):
        x=tf.keras.layers.Conv1D(filters,3,padding='causal',dilation_rate=dilation,activation='relu')(x)
        x=tf.keras.layers.SpatialDropout1D(.10)(x)
    if shortcut.shape[-1] != filters: shortcut=tf.keras.layers.Conv1D(filters,1,padding='same')(shortcut)
    return tf.keras.layers.Activation('relu')(tf.keras.layers.Add()([shortcut,x]))
def build_weather_tcn():
    inp=tf.keras.Input((LOOKBACK,len(TCN_FEATURES)),name='weather_and_residual_history'); x=inp
    for d in [1,2,4,8,16,32,64]: x=residual_block(x,64,d)
    x=tf.keras.layers.GlobalAveragePooling1D()(x); x=tf.keras.layers.Dense(64,activation='relu')(x); x=tf.keras.layers.Dropout(.20)(x)
    return tf.keras.Model(inp,tf.keras.layers.Dense(1,dtype='float32',name='scaled_next_residual')(x),name='las_vegas_weather_residual_tcn')
with tf.device('/GPU:0'):
    weather_tcn=build_weather_tcn(); weather_tcn.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse',metrics=['mae'])
    model_probe=weather_tcn(tf.zeros((1,LOOKBACK,len(TCN_FEATURES))))
assert 'GPU' in model_probe.device.upper(), f'Model probe ran on {model_probe.device}'
print('TCN GPU execution verified:',model_probe.device)
weather_tcn.summary()

TCN GPU execution verified: /job:localhost/replica:0/task:0/device:GPU:0
Model: "las_vegas_weather_residual_tcn"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 weather_and_residual_history (  [(None, 168, 23)]   0           []                               
 InputLayer)                                                                                      
                                                                                                  
 conv1d (Conv1D)                (None, 168, 64)      4480        ['weather_and_residual_history[0]
                                                                 [0]']                            
                                                                                                  
 spatial_dropout1d (SpatialDrop  (None, 168, 64)     0           ['conv1d[0][0]']              

In [8]:
# Train weather branch on Las Vegas only. Do not wrap fit in tf.device: dataset bookkeeping legitimately runs on CPU.
tcn_dir=RESULT_DIR/f'{SOURCE_CITY}_to_{TARGET_CITY}'; tcn_dir.mkdir(exist_ok=True)
callbacks=[tf.keras.callbacks.ModelCheckpoint(tcn_dir/'best_weather_tcn.keras',monitor='val_loss',save_best_only=True),tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=5,min_lr=1e-6),tf.keras.callbacks.CSVLogger(tcn_dir/'weather_tcn_history.csv')]
history=weather_tcn.fit(train_ds,validation_data=val_ds,epochs=TCN_EPOCHS,verbose=1,callbacks=callbacks)
weather_tcn.save(tcn_dir/'final_weather_tcn.keras')
joblib.dump(x_scaler,tcn_dir/'tcn_feature_scaler_las_vegas_only.joblib'); joblib.dump(r_scaler,tcn_dir/'residual_target_scaler_las_vegas_only.joblib')
(tcn_dir/'experiment_config.json').write_text(json.dumps({'source_city':SOURCE_CITY,'target_city':TARGET_CITY,'train_end':str(TRAIN_END),'validation_end':str(VALID_END),'test_period':'2023','lookback_hours':LOOKBACK,'half_life_years':HALF_LIFE_YEARS,'tcn_features':TCN_FEATURES},indent=2))

Epoch 1/100
817/817 [==============================] - 35s 35ms/step - loss: 0.2270 - mae: 0.5149 - val_loss: 0.4069 - val_mae: 0.4921 - lr: 0.0010
Epoch 2/100
817/817 [==============================] - 26s 32ms/step - loss: 0.0957 - mae: 0.3538 - val_loss: 0.1820 - val_mae: 0.3257 - lr: 0.0010
Epoch 3/100
817/817 [==============================] - 26s 32ms/step - loss: 0.0595 - mae: 0.2796 - val_loss: 0.1315 - val_mae: 0.2732 - lr: 0.0010
Epoch 4/100
817/817 [==============================] - 26s 31ms/step - loss: 0.0464 - mae: 0.2460 - val_loss: 0.1104 - val_mae: 0.2468 - lr: 0.0010
Epoch 5/100
817/817 [==============================] - 26s 31ms/step - loss: 0.0398 - mae: 0.2263 - val_loss: 0.0864 - val_mae: 0.2134 - lr: 0.0010
Epoch 6/100
817/817 [==============================] - 26s 31ms/step - loss: 0.0356 - mae: 0.2134 - val_loss: 0.0847 - val_mae: 0.2105 - lr: 0.0010
Epoch 7/100
817/817 [==============================] - 26s 31ms/step - loss: 0.0330 - mae: 0.2037 - val_loss: 0.

868

In [9]:
# Cross-city evaluation: Las Vegas weather TCN + Phoenix-only baseline → unseen Phoenix 2023 MW.
best_tcn=tf.keras.models.load_model(tcn_dir/'best_weather_tcn.keras')
pred_scaled=best_tcn.predict(target_test_ds,verbose=1).ravel()
pred_residual=r_scaler.inverse_transform(pred_scaled.reshape(-1,1)).ravel()
p=target_test_pos
pred_log=target_df.baseline_log.to_numpy()[p] + pred_residual
pred_mw=np.exp(pred_log); actual_mw=target_df.load.to_numpy()[p]
mae=float(np.mean(np.abs(actual_mw-pred_mw))); rmse=float(np.sqrt(np.mean((actual_mw-pred_mw)**2))); mape=float(np.mean(np.abs((actual_mw-pred_mw)/actual_mw))*100); r2=float(1-np.sum((actual_mw-pred_mw)**2)/np.sum((actual_mw-actual_mw.mean())**2))
metrics={'experiment':'Las Vegas weather residual TCN -> Phoenix 2023','MAE_MW':mae,'RMSE_MW':rmse,'MAPE_percent':mape,'R2':r2,'test_samples':int(len(p))}
print(json.dumps(metrics,indent=2))
pd.DataFrame({'timestamp':target_df.timestamp.iloc[p].to_numpy(),'actual_phoenix_mw':actual_mw,'phoenix_baseline_mw':np.exp(target_df.baseline_log.to_numpy()[p]),'predicted_weather_residual_log':pred_residual,'predicted_phoenix_mw':pred_mw}).to_csv(tcn_dir/'phoenix_2023_cross_city_predictions.csv',index=False)
(tcn_dir/'phoenix_2023_cross_city_metrics.json').write_text(json.dumps(metrics,indent=2))
assert pd.Timestamp(target_df.timestamp.iloc[p].min()) > VALID_END
print('Leakage check passed: all evaluated Phoenix targets are after 2022-12-31.')

137/137 [==============================] - 2s 8ms/step
{
  "experiment": "Las Vegas weather residual TCN -> Phoenix 2023",
  "MAE_MW": 119.50098257407467,
  "RMSE_MW": 168.62786104332255,
  "MAPE_percent": 1.897291265879563,
  "R2": 0.9924750040073155,
  "test_samples": 8760
}
Leakage check passed: all evaluated Phoenix targets are after 2022-12-31.


## Result folders

`two_branch_tcn_results/las_vegas/` and `two_branch_tcn_results/phoenix/` contain their independent baseline models and scalers. `two_branch_tcn_results/las_vegas_to_phoenix/` contains the Las Vegas-trained weather TCN, source-only scalers, history, configuration, Phoenix 2023 predictions, and final cross-city metrics.

For the reverse experiment, swap `SOURCE_CITY` and `TARGET_CITY`, restart the kernel, and run every cell.

# Cross-city electricity-demand forecasting results

## Experiment at a glance

| Item | Value |
|---|---|
| Source city for learned weather relationship | Las Vegas |
| Target city for transfer evaluation | Phoenix |
| Training period | 1 Jan 2016 – 31 Dec 2021 |
| Validation period | Calendar year 2022 |
| Final held-out test period | Calendar year 2023 |
| Forecast horizon | 1 hour ahead |
| Sequence lookback | 168 hours (7 days) |
| Test observations | 8,760 hourly forecasts |
| Recency weighting | Exponential, two-year half-life |
| Training objective | Mean squared error on scaled log residuals |

## Aim

Las Vegas and Phoenix have different absolute demand levels. Directly transferring a model trained to predict raw MW would therefore entangle city scale with weather response. The experiment separates the problem into:

```text
log(predicted load at t+1)
    = city-specific baseline at t+1
    + transferred weather/residual effect at t+1
```

The final load in MW is recovered as:

```text
predicted load = exp(predicted log load)
```

This lets Phoenix retain its own normal demand level, calendar profile, and long-term trend, while the Las Vegas-trained TCN supplies the learned short-term weather/residual adjustment.

## Data preparation and leakage controls

The raw hourly weather and electricity-demand CSVs for both cities cover 2016–2023. Weather timestamps were aligned to an hourly grid. Invalid physical readings were converted to missing values; weather was forward-filled only, so no future weather observation was used to fill an earlier hour.

The time split is chronological. No 2022 or 2023 target values were used to fit the baseline or TCN scalers/models. The final Phoenix test is strictly after the validation cutoff.

Recent data is weighted more heavily during training:

```text
weight = exp(-ln(2) × age_in_years / 2)
```

Thus, an observation two years older than the latest training observation has half its training weight. This accommodates gradual changes such as infrastructure expansion and demand-pattern drift without allowing future data into training.

## Layer 1 — city-specific baseline neural network

Two independent baseline MLPs were trained: one for Las Vegas and one for Phoenix. They use only calendar and trend variables—no weather variables—so they estimate each city's expected normal load.

### Baseline inputs (8)

- Hour-of-day sine and cosine
- Day-of-week sine and cosine
- Month sine and cosine
- Weekend flag
- Continuous years-since-start trend

### Baseline architecture

```text
8 inputs
  → Dense(64, ReLU)
  → Dropout(0.10)
  → Dense(32, ReLU)
  → Dense(1, float32): scaled log baseline
```

| Baseline model | Trainable parameters |
|---|---:|
| Las Vegas baseline MLP | 2,689 |
| Phoenix baseline MLP | 2,689 |

For every hourly record, the baseline output is converted back to log-load and used to calculate:

```text
residual_log = log(actual load) − log(city baseline load)
```

## Layer 2 — Las Vegas weather-residual TCN

The TCN is trained only with Las Vegas data from 2016–2021. Its target is the next-hour log residual, not the raw electricity demand. The model therefore learns the transferable short-term adjustment around the normal city-specific baseline.

### TCN input

Each forecast receives 168 previous hourly observations and 23 features per hour, giving an input shape of:

```text
(batch size, 168 hours, 23 features)
```

The 23 features are 17 weather-derived features and 6 causal residual-history features.

**Weather-derived features**

- Temperature, dew point, relative humidity, pressure
- Wind speed, wind-direction sine/cosine, visibility
- Precipitable water and hourly precipitation
- Total/opaque cloud cover
- Extraterrestrial horizontal/direct-normal radiation
- Cooling-degree and heating-degree variables
- Temperature × humidity interaction

**Residual-history features**

- Residual at the current observed hour and lags of 1, 24, and 168 hours
- 24-hour rolling residual mean and standard deviation

The residual-history terms make this a realistic one-hour-ahead operational forecast: the previous hour's actual load is known when predicting the next hour. They also mean the TCN is a **weather-plus-autoregressive-residual** model, rather than a weather-only model.

### TCN architecture

```text
Input: 168 × 23
  → 7 residual TCN blocks with dilation rates 1, 2, 4, 8, 16, 32, 64
      each block: causal Conv1D(64, kernel=3) → SpatialDropout(0.10)
                  → causal Conv1D(64, kernel=3) → SpatialDropout(0.10)
                  → residual skip connection → ReLU
  → GlobalAveragePooling1D
  → Dense(64, ReLU)
  → Dropout(0.20)
  → Dense(1, float32): next-hour scaled log residual
```

The causal convolutions ensure that each temporal representation uses only the present and earlier input hours. Dilation lets the model model short and longer delayed effects without using recurrent layers.

| TCN property | Value |
|---|---:|
| Residual blocks | 7 |
| Convolution filters | 64 |
| Kernel size | 3 |
| Dilation rates | 1, 2, 4, 8, 16, 32, 64 |
| Dropout | 0.10 in TCN blocks; 0.20 before output |
| Total/trainable parameters | 170,817 / 170,817 |

## Training process

The notebook verifies TensorFlow GPU execution with an explicit GPU matrix-multiplication probe before any model is trained. Mixed precision is enabled when configured in the GPU environment.

The TCN used Adam optimization, an initial learning rate of 0.001, learning-rate reduction on validation-loss plateau, early stopping, and best-model checkpointing. Training stopped after 41 epochs. The best validation checkpoint was at epoch 29:

| Measure at best validation epoch | Value |
|---|---:|
| Training loss | 0.01622 |
| Training MAE, scaled residual | 0.14001 |
| Validation loss | 0.04566 |
| Validation MAE, scaled residual | 0.14665 |
| Learning rate | 0.00025 |

## Cross-city reconstruction and final evaluation

For every Phoenix hour in 2023:

1. The Phoenix baseline MLP predicts Phoenix's normal log-load using calendar and trend inputs.
2. The Las Vegas-trained TCN predicts the next-hour weather/residual adjustment from Phoenix weather and previously observed Phoenix residual history.
3. The two outputs are added in log space.
4. The result is exponentiated to obtain predicted Phoenix demand in MW.

No Phoenix 2023 target values were used to train either baseline or the Las Vegas TCN. The evaluation covers the complete 2023 year.

| Final Phoenix 2023 result | Value |
|---|---:|
| MAE | **119.50 MW** |
| RMSE | **168.63 MW** |
| MAPE | **1.90%** |
| R² | **0.9925** |
| Number of forecasts | **8,760** |

## Interpretation

The 0.9925 R² and 1.90% MAPE show that the two-branch framework transferred very effectively from Las Vegas to Phoenix for this one-hour-ahead forecasting setup. The result should be presented as evidence that a weather/residual relationship learned in Las Vegas can be combined with Phoenix's separately calibrated structural baseline to forecast Phoenix demand accurately.

The correct claim is not that Phoenix demand is predicted from weather alone. Its calendar/trend baseline and observed prior residuals provide city-specific context; the cross-city component is the weather-residual TCN trained only on Las Vegas.

## Reproducibility artifacts

The result folder contains:

- `las_vegas/baseline_mlp.keras` and `phoenix/baseline_mlp.keras`
- City-specific baseline scalers
- `las_vegas_to_phoenix/best_weather_tcn.keras` and `final_weather_tcn.keras`
- Las Vegas-only TCN feature and residual scalers
- `weather_tcn_history.csv`
- `experiment_config.json`
- `phoenix_2023_cross_city_predictions.csv`
- `phoenix_2023_cross_city_metrics.json`
